# InfiniteScienceGym regeneration, part 2 of 2: generate on two T4 GPUs

Settings: Accelerator **GPU T4 x2**, *Internet* on. Inputs: the output of `isg_prepare.ipynb`
(`MODE = "prepare"`), and from the second session on, this notebook's previous output. Keep it
**private**: the repositories hold the answers.

Run with *Save Version*, *Save & Run All (Commit)*; the browser need not stay open.

* One vLLM server per GPU writes for `PER_GPU` processes at once (it batches their calls); each
  process builds one shard of the seeds (`--shard I/N`, N = 2 x `PER_GPU`; keep `PER_GPU` the same
  for the whole job).
* No repository is started after `HOURS` hours; those in progress finish. A session ends at 12 hours,
  so leave room for the slowest repository: `HOURS = 1` for a first, measuring session, `9.5` after.
* The progress lines give repositories done and tokens per second; the last cell pools the
  manifests. Questions and the audit run on CPU: `isg_prepare.ipynb` with `MODE = "finish"`.
* `ENGINE = "ollama"` is the fallback if vLLM does not start on the T4 (slower: two slots per GPU).
  The engine is recorded per repository; do not mix engines in one sample without saying so.

In [ ]:
ENGINE = "vllm"          # "vllm", or "ollama" if vLLM fails on the T4
HOURS = 1.0              # start no repository after this; 9.5 for a full session
PER_GPU = 8              # processes per GPU; N = 2 * PER_GPU shards, the same for the whole job
NUM_CTX = 16384          # tokens per sequence (vLLM --max-model-len, Ollama num_ctx), the same everywhere
VLLM_VERSION = ""        # empty: the one in plan.json
GPU_MEMORY = 0.90        # vLLM's share of each T4
OLLAMA_VERSION, OLLAMA_MODEL, OLLAMA_DIGEST = "0.40.13", "qwen3:4b-instruct-2507-fp16", "259cbeb0f33b"
OUT = "/kaggle/working"

In [ ]:
import glob, json, os, shutil, subprocess, sys, time

def sh(cmd, **kw):
    """Run a shell command, show it, stop the notebook if it fails."""
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, **kw)

def isg_venv(packages, torch_cpu, path="/tmp/isg_venv"):
    """ISG's own versions in their own environment (ISG runs model-written code)."""
    py = f"{path}/bin/python"
    if not os.path.exists(py):
        sh(f"{sys.executable} -m pip install -q uv")
        sh(f"{sys.executable} -m uv venv -q --python 3.12 {path}")
        sh(f"{sys.executable} -m uv pip install -q --python {py} {torch_cpu} --index-url https://download.pytorch.org/whl/cpu")
        sh(f"{sys.executable} -m uv pip install -q --python {py} " + " ".join(f"'{p}'" for p in packages))
    return py

def entries(path):
    with open(path) as f:
        m = json.load(f)
    return len(m.get("seconds", {})) + len(m.get("failed", {}))

def gather(sources, fs, manifests=lambda name: True):
    """Finished repositories (<seed>.json) and manifests from attached outputs into fs;
    of two copies of a manifest, the one with more entries wins."""
    os.makedirs(fs, exist_ok=True)
    for path in sorted(sources):
        name, dest = os.path.basename(path), os.path.join(fs, os.path.basename(path))
        if name.startswith("regeneration_manifest") and manifests(name):
            if not os.path.exists(dest) or entries(path) > entries(dest):
                shutil.copy(path, dest)
        elif name.split(".")[0].isdigit() and name.endswith(".json") and not os.path.exists(dest):
            shutil.copy(path, dest)
    return len(glob.glob(os.path.join(fs, "[0-9]*.json")))

In [ ]:
plans = glob.glob("/kaggle/input/**/plan.json", recursive=True)
assert plans, "add the output of isg_prepare.ipynb (MODE = 'prepare') as an input"
prep = os.path.dirname(plans[0])
plan = json.load(open(plans[0]))
VLLM_VERSION = VLLM_VERSION or plan["vllm"]
N = 2 * PER_GPU
for path in glob.glob("/kaggle/input/**/run.json", recursive=True):  # an earlier session of this job
    assert json.load(open(path))["shards"] == N, f"{path} used another number of shards: keep PER_GPU as it was"
fs = f"{OUT}/fs"
os.makedirs(f"{OUT}/logs", exist_ok=True)
mine = {f"regeneration_manifest.shard-{i}-of-{N}.json" for i in range(1, N + 1)}
done = gather([p for p in glob.glob("/kaggle/input/**/fs/*.json", recursive=True)], fs, lambda name: name in mine)
json.dump({"shards": N, "engine": ENGINE, "plan": plan}, open(f"{OUT}/run.json", "w"), indent=1)
print(f"{done} of {plan['repositories']} repositories already built; {N} shards")
shutil.copytree(f"{prep}/code", "/tmp/code", dirs_exist_ok=True)   # ISG may write next to its code
py = isg_venv(plan["isg_packages"], plan["torch_cpu"])

In [ ]:
import shlex
import urllib.request
servers, urls, logs = [], [], []
if ENGINE == "vllm":
    if not os.path.exists("/tmp/vllm_venv/bin/vllm"):
        sh(f"{sys.executable} -m uv venv -q --python 3.12 /tmp/vllm_venv")
        sh(f"{sys.executable} -m uv pip install -q --python /tmp/vllm_venv/bin/python 'vllm=={VLLM_VERSION}'"
           if VLLM_VERSION else f"{sys.executable} -m uv pip install -q --python /tmp/vllm_venv/bin/python vllm")
    version = subprocess.run(["/tmp/vllm_venv/bin/python", "-c", "import vllm; print(vllm.__version__)"],
                             capture_output=True, text=True).stdout.strip()
    serve = (f"vllm serve {prep}/model --served-model-name {plan['model']} --dtype half --max-model-len {NUM_CTX} "
             f"--gpu-memory-utilization {GPU_MEMORY} --host 127.0.0.1")
    for gpu in (0, 1):
        logs.append(f"{OUT}/logs/vllm-gpu{gpu}.log")
        servers.append(subprocess.Popen(shlex.split(f"/tmp/vllm_venv/bin/{serve} --port {8000 + gpu}"),
                                        env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)},
                                        stdout=open(logs[-1], "w"), stderr=subprocess.STDOUT))
        urls.append(f"http://127.0.0.1:{8000 + gpu}/v1")
    note = f"vLLM {version}: {serve} (one server per T4)"
else:
    sh(f"curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION={OLLAMA_VERSION} sh > {OUT}/logs/ollama_install.log 2>&1")
    for gpu in (0, 1):
        logs.append(f"{OUT}/logs/ollama-gpu{gpu}.log")
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "OLLAMA_HOST": f"127.0.0.1:{11434 + gpu}",
               "OLLAMA_NUM_PARALLEL": "2"}
        servers.append(subprocess.Popen(["ollama", "serve"], env=env, stdout=open(logs[-1], "w"),
                                        stderr=subprocess.STDOUT))
        urls.append(f"http://127.0.0.1:{11434 + gpu}")
    time.sleep(15)
    sh(f"OLLAMA_HOST=127.0.0.1:11434 ollama pull {OLLAMA_MODEL} > {OUT}/logs/pull.log 2>&1")
    note = f"Ollama {OLLAMA_VERSION}, OLLAMA_NUM_PARALLEL=2 (one server per T4)"

probe = [u + "/models" if ENGINE == "vllm" else u + "/api/version" for u in urls]
for url, server, log in zip(probe, servers, logs):
    for _ in range(180):                     # loading the model and capturing CUDA graphs take minutes
        try:
            urllib.request.urlopen(url, timeout=10)
            break
        except Exception:
            if server.poll() is not None:
                raise SystemExit(f"the server stopped. The end of {log}:\n" + open(log).read()[-3000:])
            time.sleep(10)
    else:
        raise SystemExit(f"no answer from {url} after 30 minutes; see {log}")
print("servers up:", urls, "|", note)

In [ ]:
tool = "/tmp/code/sds/tools/audit_isg.py"
procs = []
for gpu, url in enumerate(urls):
    for k in range(PER_GPU):
        i = gpu * PER_GPU + k + 1
        engine = (["--backend", "openai", "--openai-url", url, "--weights-revision", plan["revision"]]
                  if ENGINE == "vllm" else ["--backend", "ollama", "--ollama-url", url, "--ollama-model", OLLAMA_MODEL,
                                            "--ollama-digest", OLLAMA_DIGEST])
        cmd = [py, "-I", "-X", "utf8", tool, "generate", "--isg", "/tmp/code/isg", "--fs-dir", fs, "--model",
               plan["model"], "--tokenizer", f"{prep}/model", *engine, "--num-ctx", str(NUM_CTX),
               "--shard", f"{i}/{N}", "--stop-after", str(HOURS), "--machine", f"Kaggle, Tesla T4 (GPU {gpu})",
               "--note", note]
        procs.append(subprocess.Popen(cmd, stdout=open(f"{OUT}/logs/shard-{i}.log", "w"), stderr=subprocess.STDOUT))

def written():
    total = 0
    for m in glob.glob(f"{fs}/regeneration_manifest.shard-*-of-{N}.json"):
        try:
            total += sum(c.get("tokens", 0) for c in json.load(open(m)).get("model_calls", {}).values())
        except ValueError:                   # being written right now
            pass
    return total

t0, start, tokens0, last = time.time(), done, written(), 0.0
while any(p.poll() is None for p in procs):
    time.sleep(30)
    if time.time() - last < 600 and any(p.poll() is None for p in procs):
        continue
    last = time.time()
    now, hours = len(glob.glob(f"{fs}/[0-9]*.json")), (time.time() - t0) / 3600
    print(f"{hours:.1f} h: {now} of {plan['repositories']} repositories ({now - start} this session), "
          f"{sum(p.poll() is None for p in procs)} processes running, about "
          f"{(written() - tokens0) / max(hours * 3600, 1):.0f} tokens/s in finished repositories", flush=True)
for server in servers:
    server.terminate()
print("exit codes:", [p.returncode for p in procs], "(non-zero: see logs/shard-*.log)")

In [ ]:
sys.path.insert(0, "/tmp/code/sds/tools")
import audit_isg
pooled = audit_isg.merge_manifests(audit_isg.manifest_paths(fs))
print(json.dumps({k: v for k, v in pooled.items() if k not in ("built", "failed")}, indent=1))
print("failed:", pooled["failed"])